# Milestone 4 lab, notebook 6 of 8: Quarantine

## Where we are

| Notebook | What you do |
|---|---|
| 1. A rule lives in dbt | Move a check out of a notebook and into a file that dbt runs |
| 2. How serious is it | Tell dbt which failures stop the night. Keep the rows a test returns |
| 3. Test the SQL, not the data | Check your own query on four rows typed by hand |
| 4. A gate, and where it stands | Run one rule in three places, and see what each place can see |
| 5. Warn, block or set aside | Decide what a failure does. Then block a real run in Airflow |
| **6. Quarantine** (you are here) | Set the bad rows aside with a reason, and let the good rows go on |
| 7. Test the tests | Plant a fault, and prove its gate catches it. One check in four tools |
| 8. The production version | Run the whole week through Airflow. See what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import (
    backup, change, money, prepare_week, restore, run_from, show_report, sql,
)

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August, through three gates.
Sunday night ran 47 tests. 1 of 6 stores can be paid.


## Step 1: One bad row, six stores

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, and what happens when it fails.

On Monday 24 August one row in store `S03`'s file was wrong: a refund of $9.49 with a positive amount. That night the six stores sent 610 good lines and one bad one.

Blocking would keep six managers waiting for one row. Warning would publish $796.18 again. There is a third way.

**Quarantine**: the bad rows are set aside in their own table, with the reason, and the good rows carry on.

## Step 2: The row, and its reasons

The clean gate moves a bad row out of `clean.sales_lines` and into `quarantine.sales_lines`. It adds what it knows: which test, why, and when.

In [2]:
sql("""
    SELECT line_id, store_id, file_day, product_name, amount_cents, check_key, reason
    FROM quarantine.sales_lines
    WHERE NOT is_a_copy
    ORDER BY file_day, line_id, check_key
""")

,line_id,store_id,file_day,product_name,amount_cents,check_key,reason
0,S03-0824-000008,S03,2026-08-24,Olive oil 500ml,949,line_adds_up,A line adds up
1,S03-0824-000008,S03,2026-08-24,Olive oil 500ml,949,refund_gives_money_back,A refund gives money back
2,S05-0827-000020,S05,2026-08-27,Bananas,152220,believable_quantity,A weight is a believable weight
3,S02-0830-000142,S02,2026-08-30,Coffee beans 250g,899,inside_opening_hours,Nothing is sold while the store is closed
4,S02-0830-000143,S02,2026-08-30,Oat milk 1L,698,inside_opening_hours,Nothing is sold while the store is closed
5,S02-0830-000144,S02,2026-08-30,"Eggs, 12",519,inside_opening_hours,Nothing is sold while the store is closed
6,S02-0830-000145,S02,2026-08-30,Dark chocolate 100g,279,inside_opening_hours,Nothing is sold while the store is closed


Seven rows for six lines. The refund is there **twice**, with two reasons: two different tests returned it. A row can break more than one rule, and the table keeps every reason.

The rows a check refused are called **rejected rows**, or rejects. A quarantine table is where they wait.

Is the refund still in the clean table?

In [3]:
sql("""
    SELECT 'clean.sales_lines' AS in_table, count(*) AS rows FROM clean.sales_lines WHERE line_id = 'S03-0824-000008'
    UNION ALL
    SELECT 'quarantine.sales_lines', count(DISTINCT line_id) FROM quarantine.sales_lines WHERE line_id = 'S03-0824-000008'
    UNION ALL
    SELECT 'raw.sales_lines', count(*) FROM raw.sales_lines WHERE line_id = 'S03-0824-000008'
""")

,in_table,rows
0,clean.sales_lines,0
1,quarantine.sales_lines,1
2,raw.sales_lines,1


Gone from clean, kept in quarantine, and still in raw. Nothing was deleted. A row that is set aside can always be looked at, and brought back.

## Step 3: Set aside, and the day is right again

Some rows are bad in a simple way: they are extra copies. On Tuesday store `S01` sent one receipt twice. On Friday store `S02` sent its whole file twice.

In [4]:
sql("""
    SELECT store_id, file_day, count(DISTINCT line_id) AS copies_set_aside, sum(amount_cents) AS cents
    FROM quarantine.sales_lines
    WHERE is_a_copy
    GROUP BY store_id, file_day
    ORDER BY file_day
""")

,store_id,file_day,copies_set_aside,cents
0,S01,2026-08-25,4,1807
1,S02,2026-08-28,167,103428


In [5]:
sql("""
    SELECT store_id, sales_date, lines_cents, till_cents, rows_set_aside, status
    FROM output.store_day_status
    WHERE (store_id = 'S01' AND sales_date = DATE '2026-08-25')
       OR (store_id = 'S02' AND sales_date = DATE '2026-08-28')
    ORDER BY sales_date
""")

,store_id,sales_date,lines_cents,till_cents,rows_set_aside,status
0,S01,2026-08-25,81977,81977,4,ok
1,S02,2026-08-28,103428,103428,167,ok


The first copy of each line stayed. The extra copies went aside. And what is left adds up to the tills, to the cent: `status` is `ok`.

For these two days, quarantine did not only stop the damage. It repaired the number. Marta Lindqvist of `S01` can be paid.

## Step 4: When the day is not right again

Most rows are not copies. Taking out the bad refund does not make `S03`'s Monday right. The refund really happened: $9.49 really went back to a customer. With the row set aside, that money is simply not counted.

In [6]:
sql("""
    SELECT store_id, sales_date, lines_cents, till_cents, lines_cents - till_cents AS difference_cents,
           rows_set_aside, cents_set_aside, problem, status, reasons
    FROM output.store_day_status
    WHERE status <> 'ok'
    ORDER BY sales_date, store_id
""")

,store_id,sales_date,lines_cents,till_cents,difference_cents,rows_set_aside,cents_set_aside,problem,status,reasons
0,S03,2026-08-24,78669,77720,949,1,949,rows_set_aside,do not pay yet,A line adds up; A refund gives money back
1,S04,2026-08-25,0,58742,-58742,0,0,no_sales_file,do not pay yet,NaN
2,S05,2026-08-27,57932,58084,-152,1,152220,rows_set_aside,do not pay yet,A weight is a believable weight
3,S06,2026-08-29,97826,99623,-1797,3,1797,rows_set_aside,do not pay yet,Every sale is for a product that exists
4,S02,2026-08-30,88135,90530,-2395,4,2395,rows_set_aside,do not pay yet,Nothing is sold while the store is closed


`S03` on Monday is still $9.49 away from its tills. The pipeline does not guess. It does not flip the sign of the refund. It publishes what it is sure of, and says what is missing.

That is a **partial load**: a load where the good rows went on and the bad ones did not. A partial load is only honest if **the output says what it is missing**. This table is how it says so, and the bonus report reads it.

In [7]:
show_report()

Lotus Retails. Store managers' bonus, week of Mon 2026-08-24
Net sales from Mon 2026-08-24 to Sun 2026-08-30

  Store                           Manager              Net sales  Bonus (1%)   Pay?
  S01  Lotus Ashby High Street    Marta Lindqvist      $5,153.25      $51.53   yes
  S02  Lotus Ashby Market Square  Dana Okafor          $5,724.68           -   DO NOT PAY YET
         Sun 08-30: 4 rows set aside ($23.95): Nothing is sold while the store is closed
  S03  Lotus Calder Bridge        Tomas Reyes          $5,097.11           -   DO NOT PAY YET
         Mon 08-24: 1 row set aside ($9.49): A line adds up; A refund gives money back
  S04  Lotus Calder Station       Ines Marchetti       $4,585.57           -   DO NOT PAY YET
         Tue 08-25: no sales file arrived. The tills counted $587.42
  S05  Lotus Fenwick Green        Samir Haddad         $3,591.53           -   DO NOT PAY YET
         Thu 08-27: 1 row set aside ($1,522.20): A weight is a believable weight
  S06  Lotus Fenwick 

Read it as the head of retail operations. One store: yes. Five stores: not yet, and for each the day, the number of rows, the money and the rule. This is the first report in the project that answers her second sentence: *which one, and why?*

## Step 5: A whole file set aside

Some things cannot be taken apart row by row. A supplier's file with a renamed column has no good rows to keep. The landing gate sets the **whole file** aside.

In [8]:
sql("SELECT file_name, check_key, reason FROM ops.refused_files ORDER BY file_name")

,file_name,check_key,reason
0,supplier_prices_SUP-B_2026-08-25.csv,file_is_missing_a_column,An expected column is missing: unit_cost
1,supplier_prices_SUP-C_2026-08-28.csv,supplier_contract,"The contract is broken: unit_cost: 0 decimals, not 2: dollars are written like 2.10"


A table for things that could not be processed at all is called a **dead-letter table**. The name comes from the post office: a letter that cannot be delivered goes to the dead-letter office, and waits there for someone to look.

And the good rows that "carry on"? For a price file, they are the rows of the **last good file**.

In [9]:
written = sql("SELECT unit_cost FROM raw.supplier_prices WHERE product_id = 'P-024' AND price_date = '2026-08-28'").loc[0, "unit_cost"]
trusted = sql("""
    SELECT price_date, unit_cost_cents FROM clean.supplier_prices
    WHERE product_id = 'P-024' ORDER BY price_date DESC LIMIT 1
""")
on_the_list = sql("""
    SELECT max(unit_cost_cents) AS cents FROM output.replenishment_list
    WHERE product_id = 'P-024' AND for_date = DATE '2026-08-29'
""").loc[0, "cents"]

print("Olive oil 500ml, P-024")
print("  written in the refused file of 28 August:", written, " -> read as dollars:", money(int(written) * 100))
print("  the newest cost in the clean table is from", trusted.loc[0, "price_date"], "and is", money(trusted.loc[0, "unit_cost_cents"]))
print("  the cost on the order list for 29 August:", money(on_the_list))

Olive oil 500ml, P-024
  written in the refused file of 28 August: 655  -> read as dollars: $655.00
  the newest cost in the clean table is from 2026-08-21 and is $6.55
  the cost on the order list for 29 August: $6.55


The refused file wrote `655`, meaning cents. Read as dollars that is $655.00 for a bottle of olive oil. The order list uses $6.55, from the file of the week before.

## Step 6: Too many bad rows

One bad row in a hundred is a bad row. Thirty bad rows in a hundred is a bad **file**, and the seventy "good" ones cannot be trusted either.

So quarantine has a limit: when more than **5%** of one store's file is set aside, the whole file is set aside.

Plant that. Give every first line of every receipt in one clean file a negative quantity. Keep a copy of the raw table first, so you can put it back.

In [10]:
backup("raw.sales_lines")
change("""
    UPDATE raw.sales_lines SET quantity = '-' || quantity
    WHERE _file_name = 'sales_lines_S03_2026-08-20.csv' AND line_type = 'sale' AND line_no = '1'
""")
run_from("clean")

Kept a copy of raw.sales_lines.


clean:        7 clean tables rebuilt. sales_lines has 24,384 rows.


gate clean: 39 tests, 29 passed.
    set aside     171  One item line is stored once
    set aside      37  A line adds up
    set aside       3  Every sale is for a product that exists
    set aside      37  A refund gives money back
    warning         6  The price charged is the price valid that day
    set aside       4  Nothing is sold while the store is closed
    set aside       1  A weight is a believable weight
    warning         1  A store's day is near its normal day
    warning         1  A snapshot has a row for every day
    set aside       1  One person is one member
    rows set aside: 298 in sales_lines, 3 in raw_sales_lines, 1 in members
build:        staging.daily_store_sales has 207 rows. Nobody reads staging.


gate publish: 5 tests, 4 passed.
    warning         6  Every open store has a day we can pay from
publish:      daily_store_sales has 207 rows. replenishment_list has 234 rows for tomorrow. 6 store days are not ready to pay.


Look at the clean gate's lines: `A refund gives money back` now sets 37 rows aside, not 1. Thirty-six of them are yours.

In [11]:
sql("""
    SELECT check_key, count(DISTINCT line_id) AS lines
    FROM quarantine.sales_lines
    WHERE store_id = 'S03' AND file_day = DATE '2026-08-20'
    GROUP BY check_key ORDER BY lines DESC
""")

,check_key,lines
0,too_many_bad_rows,85
1,line_adds_up,36
2,refund_gives_money_back,36


In [12]:
sql("""
    SELECT (SELECT count(*) FROM raw.sales_lines WHERE _file_name = 'sales_lines_S03_2026-08-20.csv') AS lines_in_the_file,
           (SELECT count(*) FROM clean.sales_lines WHERE store_id = 'S03' AND file_day = DATE '2026-08-20') AS lines_left_in_clean,
           (SELECT status FROM output.store_day_status WHERE store_id = 'S03' AND sales_date = DATE '2026-08-20') AS status
""")

,lines_in_the_file,lines_left_in_clean,status
0,121,0,do not pay yet


The 36 planted lines broke two rules. The file has 121 lines, so that is far more than 5%. The gate then set the other 85 lines aside too, with the reason `too_many_bad_rows`. Nothing of that file is left in the clean table, and the store's day says "do not pay yet".

Put the raw table back.

In [13]:
restore("raw.sales_lines")

Put back: raw.sales_lines. Sunday's night ran again, and the fault is gone.


## Step 7: People, set aside by number

One more table. The duplicate member of milestone 3 is set aside too: the newer of the two accounts.

In [14]:
sql("SELECT * FROM quarantine.members")

,member_id,check_key,reason,set_aside_on
0,M-0304,one_person_once,One person is one member,2026-08-30


A member number, the test, the reason, the day. **No name and no email.** A quarantine table is a copy of data in a new place, read by other people. Personal data does not travel there.

## Step 8: The same fault, three ways

Take the bad refund of `S03`, and the three things a gate could do with it.

| | Warn | Set aside | Block |
|---|---|---|---|
| What is published for `S03`, Monday | $796.18 | $786.69, marked "do not pay yet" | Nothing new, for any store |
| The other five stores | Published | Published | Wait |
| Is the wrong number shown? | Yes | No. It is marked | No |
| Who must act | Nobody has to | The store sends the refund again | Somebody, tonight |

## What you now know

- **Quarantine**: bad rows are set aside in their own table with the reason, and the good rows carry on.
- **Rejected rows** are the rows a check refused. Nothing is deleted: they stay in raw and wait in quarantine.
- A **dead-letter table** holds what could not be processed at all, such as a whole file that was refused.
- A **partial load** is honest only when the output says what it is missing.
- More than 5% of a file set aside, and the whole file is refused.
- Set aside is not "repaired". Extra copies repaired two days. Five other days are still wrong, and say so.

**Say it in an interview:** "A bad refund in one store should not stop pay for five others. Our clean gate moved failing rows to a quarantine table with the test and the reason, and published the rest. A status table said, per store and day, what was set aside, and the bonus report marked that store 'do not pay yet'. Above 5% of a file, we refused the whole file."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "Warn or block", and [Python: Validation, Contracts & Error Handling](https://www.d8loop.com/modules/knowledge-corner/python/validation-contracts-error-handling).

Next: [notebook 7](07_test_the_tests.ipynb). Forty-seven tests guard the night. Prove they work.